In [14]:
!git -C /Users/colelacroix/math config http.postBuffer 524288000
!git -C /Users/colelacroix/math push origin main

Enumerating objects: 45, done.
Counting objects: 100% (45/45), done.
Delta compression using up to 8 threads
Compressing objects: 100% (39/39), done.
Writing objects: 100% (40/40), 3.28 MiB | 9.05 MiB/s, done.
Total 40 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
error: RPC failed; curl 55 Recv failure: Connection reset by peer
send-pack: unexpected disconnect while reading sideband packet
fatal: the remote end hung up unexpectedly
Everything up-to-date


In [15]:
!git -C /Users/colelacroix/math status -sb

## main...origin/main [ahead 1]
 M library_upate.ipynb


In [10]:
import os
os.chdir("/Users/colelacroix/math")

In [11]:
import sys
!{sys.executable} -m pip install pyyaml

In [12]:
import yaml, os, glob, subprocess, re

ROOT = "/Users/colelacroix/math"
EXCLUDE = {"inventory", "_build", ".git", ".ipynb_checkpoints", "__pycache__"}
GITIGNORE = """\
# Managed by library_update.ipynb
inventory/
_build/
.ipynb_checkpoints/
__pycache__/
*.pyc
.DS_Store
"""
MAX_MB = 50  # GitHub warns at 50 MB and rejects files over 100 MB

def clean_title(name):
    stripped = re.sub(r"^\d+\s*[-_.)]?\s*", "", name)
    return stripped.replace("_", " ").replace("-", " ").strip().title()

def toc_for(folder):
    """Notebooks in this folder, plus a section for each subfolder that has any."""
    items = [{"file": os.path.relpath(nb, ROOT)}
             for nb in sorted(glob.glob(os.path.join(folder, "*.ipynb")))]
    for entry in sorted(os.listdir(folder)):
        full = os.path.join(folder, entry)
        if os.path.isdir(full) and not entry.startswith(".") and entry not in EXCLUDE:
            children = toc_for(full)
            if children:
                items.append({"title": clean_title(entry), "children": children})
    return items

def rebuild_toc():
    toc = [{"file": "index.md"}] + toc_for(ROOT)
    config = {
        "version": 1,
        "project": {
            "id": "af601852-cbc3-4d41-9731-bb9a87aa495e",
            "title": "Math Notes",
            "description": "Notes and worked examples",
            "github": "https://github.com/coletlacroix/math",
            "toc": toc,
        },
        "site": {"template": "book-theme"},
    }
    with open(os.path.join(ROOT, "myst.yml"), "w") as f:
        yaml.dump(config, f, sort_keys=False, allow_unicode=True)
    print("myst.yml rebuilt")

def update_gitignore():
    """Add any missing GITIGNORE lines to .gitignore, keeping what's already there."""
    path = os.path.join(ROOT, ".gitignore")
    existing = open(path).read().splitlines() if os.path.exists(path) else []
    missing = [l for l in GITIGNORE.splitlines() if l and l not in existing]
    if missing:
        with open(path, "a") as f:
            f.write(("\n" if existing else "") + "\n".join(missing) + "\n")

def git(*args):
    r = subprocess.run(["git", "-C", ROOT, *args], capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())
    return r

def push(message):
    update_gitignore()
    rebuild_toc()
    git("add", "-A")

    # Stop before committing if anything staged is too big for GitHub
    staged = subprocess.run(["git", "-C", ROOT, "diff", "--cached", "--name-only", "-z",
                             "--diff-filter=AM"],
                            capture_output=True, text=True).stdout.split("\0")
    big = [(p, os.path.getsize(os.path.join(ROOT, p)) / 1e6)
           for p in staged if p and os.path.isfile(os.path.join(ROOT, p))
           and os.path.getsize(os.path.join(ROOT, p)) > MAX_MB * 1e6]
    if big:
        git("reset", "-q")
        print(f"\nStopped: these files are over {MAX_MB} MB. Add them to GITIGNORE or move them out of {ROOT}:")
        for p, mb in big:
            print(f"  {mb:8.1f} MB  {p}")
        return

    if git("commit", "-m", message).returncode == 0:
        git("push", "origin", "main")

In [13]:
# Leave new message for each push

push("restart")

myst.yml rebuilt

[main 8167585] restart
 50 files changed, 65075 insertions(+), 28125 deletions(-)
 rename {02_Algebra_and_Trigonometry => 01_Algebra_and_Trigonometry}/1_prerequisites.ipynb (100%)
 delete mode 100644 01_Think_Bayes_(Downey)/01_probability.ipynb
 delete mode 100644 01_Think_Bayes_(Downey)/03_distributions.ipynb
 delete mode 100644 01_Think_Bayes_(Downey)/04_estimating_proportions.ipynb
 delete mode 100644 01_Think_Bayes_(Downey)/05_estimating_counts.ipynb
 delete mode 100644 01_Think_Bayes_(Downey)/06_odds_and_addends.ipynb
 delete mode 100644 01_Think_Bayes_(Downey)/07_min_max_mix.ipynb
 delete mode 100644 01_Think_Bayes_(Downey)/08_poisson processes.ipynb
 delete mode 100644 01_Think_Bayes_(Downey)/09_decision_analysis.ipynb
 delete mode 100644 01_Think_Bayes_(Downey)/10_testing.ipynb
 delete mode 100644 01_Think_Bayes_(Downey)/11_comparison.ipynb
 delete mode 100644 01_Think_Bayes_(Downey)/12_classification.ipynb
 delete mode 100644 01_Think_Bayes_(Downey)/13_infere